In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install auto-round --break-system-packages

In [ ]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [ ]:
import os

import torch
from auto_round import AutoRound, AutoScheme, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from torch import nn
from transformers import AutoModelForImageTextToText, AutoProcessor

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    print(f"Total GPUs Detected: {gpu_count}\n")

    for idx in range(gpu_count):
        props = torch.cuda.get_device_properties(idx)
        print(f"--- GPU {idx} ---")
        print(f"  Name: {props.name}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(f"  Allocated VRAM: {torch.cuda.memory_allocated(idx) / (1024**3):.2f} GB")
        print(f"  Free VRAM: {(props.total_memory - torch.cuda.memory_reserved(idx)) / (1024**3):.2f} GB\n")
else:
    print("No CUDA-compatible GPUs found.")

In [ ]:
notebook_login()

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"
HF_USER = "your_name"               # Replace with your Hugging Face username
OUTPUT_BASE_DIR = "./AutoRound-4b"
LOCAL_PATH = "./local_model-4b"

In [ ]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

In [ ]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer

In [ ]:
def get_qwen3_5_shared_layers(
    model: nn.Module,
    skip_vision_towers: bool = True,
) -> list[list[str]]:
    """Extracts all fused kernel layer groups that must share identical bit-widths

    to prevent vLLM loader assertion errors (AssertionError in load_merged_column_weight).
    """
    FUSION_RULES = [
        # Standard MLP: gate_proj and up_proj are fused
        ("mlp", ("gate_proj", "up_proj")),
        # Full Self-Attention: Q, K, V projections are fused
        ("self_attn", ("q_proj", "k_proj", "v_proj")),
        ("attn", ("q_proj", "k_proj", "v_proj")),
        # Gated DeltaNet Linear Attention: All 4 in-projections are merged together
        ("linear_attn", ("in_proj_qkv", "in_proj_z", "in_proj_b", "in_proj_a")),
    ]

    all_modules = dict(model.named_modules())
    shared_layers: list[list[str]] = []
    processed_parents: set[str] = set()

    for full_name, module in all_modules.items():
        if not isinstance(module, nn.Linear):
            continue

        if skip_vision_towers and any(p in full_name for p in ("visual.", "merger.")):
            continue

        parent_name, leaf_name = full_name.rsplit(".", 1) if "." in full_name else ("", full_name)
        parent_type = parent_name.rsplit(".", 1)[-1] if parent_name else ""

        if parent_name and parent_name not in processed_parents:
            # Check attention, linear attention, and MLP patterns
            for target_parent, siblings in FUSION_RULES:
                if parent_type == target_parent and leaf_name in siblings:
                    found_siblings = [
                        f"{parent_name}.{sib}"
                        for sib in siblings
                        if f"{parent_name}.{sib}" in all_modules
                    ]
                    if len(found_siblings) >= 2:
                        shared_layers.append(found_siblings)
                        processed_parents.add(parent_name)
                    break

            # Handle MoE expert layers if running a MoE variant
            if "experts" in parent_name and leaf_name in ("gate_proj", "up_proj"):
                found_siblings = [
                    f"{parent_name}.{sib}"
                    for sib in ("gate_proj", "up_proj")
                    if f"{parent_name}.{sib}" in all_modules
                ]
                if len(found_siblings) >= 2:
                    shared_layers.append(found_siblings)
                    processed_parents.add(parent_name)

    return shared_layers

In [ ]:
shared_layers = get_qwen3_5_shared_layers(model, skip_vision_towers=True)

print(f"Auto-detected {len(shared_layers)} fused projection groups for vLLM.")

In [ ]:

# Configure AutoScheme for dynamic mixed-precision layer allocation
mixed_scheme = AutoScheme(
    avg_bits=3.5,                               # Target average bit-width across all quantized layers in the model
    options=["W3A16","W4A16"],                  # Candidate precision schemes evaluated by the dynamic programming solver
    ignore_scale_zp_bits=True,                  # Excludes metadata overhead (scales and zero-points) from the average bit calculation
    shared_layers=shared_layers,                # Enforces identical bit-widths on fused kernels (MLP gate/up, QKV) to prevent vLLM loader assertions
)

# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1200,                                 # Optimization iterations per block; high accuracy (Production grade)
    group_size=32,                              # Number of weights sharing one scale. Cannot go below 32 (vLLM's GPTQ kernel won't run it). Only use 16 if targeting NVFP4.
    sym=True,                                   # Symmetric quantization; enables faster compute and broader engine compatibility
    gradient_accumulate_steps=4,                # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop    
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                                # Target pretrained model to be quantized
    tokenizer=tokenizer,                        # Tokenizer used to process text calibration samples
    processor=processor,                        # Processor for multimodal models (vision/audio), if present
    scheme=mixed_scheme,                        # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,                     # Applies the algorithm-specific optimization settings defined above
    nsamples=512,                               # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=2,                               # Batch size for calibration; keep < 8 to avoid slow execution/high memory overhead
    seqlen=4096,                                # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,                    # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,                  # Compiles forward passes using torch.compile to accelerate calibration (JIT acceleration)
    quant_nontext_module=False,                 # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision (Crucial for VLM accuracy)
    ignore_layers="mtp,visual",                 # Excludes MTP, and visual modules from quantization (Add linear_attn if needed)
    layer_config={
        "mtp": {"data_type": "bfloat16"},       # Keeps the MTP module in BF16 precision
        "mtp.fc": {"data_type": "bfloat16"},    # Keeps the MTP fully connected layer in BF16 precision
    },                                          # Preserves BF16 precision for the specified MTP layers
)

In [ ]:
# Quantize once and export to disk across all major loader schemas
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Serializes weights to 3 formats simultaneously to guarantee compatibility:
    # - 'auto_round': Reference AutoRound format (vLLM GPTQ/Marlin, Transformers / native loader)
    # - 'auto_gptq': AutoGPTQ tensor layout (vLLM GPTQ/Marlin, TGI)
    # - 'llm_compressor': Compressed-tensors schema (vLLM / Neural Magic)
    format="llm_compressor",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)

In [ ]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            delete_patterns=[
                "*.safetensors",
                "*.bin",
                "*.pt",
                "*.gguf",
                "*.json",
                "*.model",
                "*.onnx",
            ],
            commit_message="Uploaded or Updated model weights and config",
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [ ]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-4b-w3g32"),
        f"{base_name}-WNA16-3.5Bits",
        hf_token,
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")